# Polymarket Static Dataset Research

Notebook de investigación para construir una primera recogida estática de datos de Polymarket y fuentes relacionadas, con el objetivo de analizarlos después como si fueran un dataset cerrado.

Objetivo de esta primera pasada:

1. Definir qué universo de mercados vamos a muestrear.
2. Determinar qué señales cruzadas aportan más información para entender Polymarket.
3. Diseñar una extracción inicial reproducible que luego podamos convertir en dataset analizable.
4. Evitar desde el principio capturar ruido que no ayude a análisis, modelado o validación.

## Hipótesis de investigación

La primera recogida de datos no busca automatizar nada; busca descubrir qué variables tienen valor explicativo cuando las cruzamos entre fuentes.

Hipótesis a probar:

- Los mercados con `resolutionSource` más concreto y verificable deberían tener menos ambigüedad en resolución.
- La combinación de `volume`, `liquidity`, spread y profundidad del book debe separar mercados informativos de mercados ruidosos.
- Los trades y holders recientes deberían concentrarse en un subconjunto pequeño de mercados realmente relevantes.
- Las señales on-chain de Polygon y UMA deberían explicar mejor la resolución que la metadata aislada.
- Si una referencia externa de Chainlink o una mención en X contradice al mercado, eso puede ser una señal útil para priorizar revisión manual.

La idea es construir un dataset estático inicial con suficiente variedad para poder hacer clustering, ranking de señales, detección de anomalías y búsqueda de predictores simples.

## Qué datos vamos a recopilar en la primera pasada

La primera recogida se organiza por capas. No queremos descargar todo Polymarket; queremos un corte representativo y enriquecido.

### 1. Universo base de mercados: Gamma API
- `events` y `markets` para descubrir mercados activos.
- `id`, `slug`, `conditionId`, `clobTokenIds`, `title`, `description`, `resolutionSource`.
- `active`, `closed`, `liquidity`, `volume`, `volume24hr`, `bestBid`, `bestAsk`, `lastTradePrice`.
- `tags`, `related_tags` y `tag_id` solo para segmentar el universo inicial.

### 2. Serie de precios y microestructura: CLOB API
- `prices-history` para una ventana estática por mercado, por ejemplo `1h` y `1d`.
- `book` para una foto de profundidad del mercado.
- `prices` en lote para obtener precios actuales de varios token IDs.
- `midpoint` y `spread` solo si queremos comparar contra el cálculo derivado.

### 3. Participación y actividad: Data API
- `trades` para capturar la actividad reciente del mercado.
- `holders` para medir concentración y distribución de posiciones.
- `activity` para ver la historia resumida de la wallet en un mercado.

### 4. Capa on-chain: Polygon + contratos de Polymarket
- Eventos del CTF Exchange para ver actividad de ejecución y settlement.
- Eventos del UMA Adapter y UMA Optimistic Oracle para entender el flujo de resolución.
- Logs de contratos de Polymarket en Polygon como evidencia estructural de lo ocurrido on-chain.

### 5. Capa externa de validación
- Chainlink para contrastar precios o referencias cuando el mercado depende de un activo o un dato verificable.
- X para menciones, contexto público y posibles señales narrativas si hay acceso suficiente.

La primera versión del dataset estático debería contener una muestra de mercados activos, una muestra de mercados ya resueltos y, cuando aplique, una muestra de mercados con resolución disputada. Eso nos permite comparar comportamiento antes, durante y después de la resolución.

## Cómo vamos a construir el dataset estático

La primera recogida no debe ser una foto aleatoria. Debe ser una muestra diseñada para comparar comportamientos distintos.

### Cohortes mínimas

- **Cohorte A: mercados activos líquidos**. Sirve para estudiar microestructura, spread y actividad real.
- **Cohorte B: mercados resueltos**. Sirve para estudiar resolución, señal previa y calidad del `resolutionSource`.
- **Cohorte C: mercados disputados o ambiguos**. Sirve para estudiar discrepancias entre mercado, UMA y fuentes externas.
- **Cohorte D: mercados de baja liquidez**. Sirve como grupo de control para separar ruido de señal.

### Ventanas de captura por mercado

- Snapshot de metadata de Gamma en el momento de la recogida.
- Histórico corto de precios, por ejemplo `1h` y `1d`.
- Foto del order book y, si interesa, niveles top 10-20.
- Trades recientes con una ventana fija.
- Top holders y actividad asociada a wallets relevantes.
- Si el mercado ya resolvió, logs on-chain alrededor del evento de resolución.

### Capas de datos en el dataset

- **Tabla de mercado**: una fila por mercado con metadata y agregados.
- **Tabla de token**: una fila por token/outcome con precio, book y series temporales.
- **Tabla de trades**: una fila por trade observado.
- **Tabla de holders**: una fila por holder relevante.
- **Tabla de actividad on-chain**: eventos de Polygon, UMA y, cuando aplique, Chainlink o señales externas.
- **Tabla externa**: referencias desde X u otras fuentes públicas con texto, timestamp y link.

Esta estructura nos permite hacer análisis estático, clustering, rankings de señal, y después entrenar modelos o reglas sin depender de reconsultas continuas.

In [6]:
import json
import os
import sys
from pathlib import Path
from pprint import pprint

NOTEBOOK_ROOT = Path.cwd().resolve()
BACKEND_ROOT = None
for candidate in (
    NOTEBOOK_ROOT,
    NOTEBOOK_ROOT / "backend",
    NOTEBOOK_ROOT.parent,
    NOTEBOOK_ROOT.parent / "backend",
):
    if (candidate / "app").exists():
        BACKEND_ROOT = candidate
        break

if BACKEND_ROOT is None:
    BACKEND_ROOT = NOTEBOOK_ROOT

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.services import PolygonClient, PolymarketClient

polymarket = PolymarketClient()
polygon = PolygonClient()

SNAPSHOT_CONFIG = {
    "market_sample_size": 50,
    "history_windows": ["1h", "1d"],
    "trade_limit": 100,
    "holders_limit": 20,
    "activity_limit": 50,
}

OPTIONAL_LOG_ADDRESS = os.getenv("POLYGON_LOG_ADDRESS")
OPTIONAL_LOG_TOPIC = os.getenv("POLYGON_LOG_TOPIC")


def normalize_collection(payload, keys=("events", "markets", "results", "data")):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in keys:
            value = payload.get(key)
            if isinstance(value, list):
                return value
        for value in payload.values():
            if isinstance(value, list):
                return value
    return []


print("Clientes y helpers listos para una captura estatica de dataset")

Clientes y helpers listos para una captura estatica de dataset


In [8]:
# 1) Descubrimiento del universo estatica de mercados activos
events_response = await polymarket.get_events(
    query_params={
        "active": "true",
        "closed": "false",
        "limit": SNAPSHOT_CONFIG["market_sample_size"],
        "offset": 0,
        "order": "volume_24hr",
        "ascending": "false",
    }
)

events_payload = events_response.json()
events = normalize_collection(events_payload)

print(type(events), len(events))
pprint(events[0] if events else {})

<class 'list'> 50
{'active': True,
 'archived': False,
 'automaticallyActive': True,
 'closed': False,
 'commentCount': 854,
 'competitive': 0.9699546837171767,
 'createdAt': '2024-12-31T16:02:31.965903Z',
 'creationDate': '2024-12-31T18:51:45.506002Z',
 'cumulativeMarkets': False,
 'cyom': False,
 'deploying': False,
 'description': 'This market will resolve to "Yes" if MicroStrategy sells any '
                'of its Bitcoin by December 31, 2025, 11:59 PM ET. Otherwise, '
                'this market will resolve to "No".\n'
                '\n'
                'The primary resolution source for this market will be '
                'information from MSTR and on-chain data, however a consensus '
                'of credible reporting will also be used.',
 'enableNegRisk': False,
 'enableOrderBook': True,
 'endDate': '2027-01-01T05:00:00Z',
 'estimateValue': False,
 'eventMetadata': {'context_requires_regen': True},
 'featured': True,
 'featuredOrder': 17,
 'gmpChartMode': 'default',

In [9]:
# 2) Extraer solo lo util de Gamma

def extract_market_insight(event):
    markets = event.get("markets", []) or []
    rows = []
    for market in markets:
        raw_token_ids = market.get("clobTokenIds") or []
        if isinstance(raw_token_ids, str):
            try:
                raw_token_ids = json.loads(raw_token_ids)
            except Exception:
                raw_token_ids = [raw_token_ids]
        elif not isinstance(raw_token_ids, list):
            raw_token_ids = [raw_token_ids]

        token_ids = [token_id for token_id in raw_token_ids if token_id]
        rows.append(
            {
                "event_id": event.get("id"),
                "event_slug": event.get("slug"),
                "market_id": market.get("id"),
                "slug": market.get("slug"),
                "title": market.get("question") or market.get("title"),
                "conditionId": market.get("conditionId"),
                "clobTokenIds": token_ids,
                "resolutionSource": market.get("resolutionSource") or event.get("resolutionSource"),
                "description": market.get("description") or event.get("description"),
                "active": market.get("active"),
                "closed": market.get("closed"),
                "liquidity": market.get("liquidity"),
                "volume": market.get("volume"),
                "volume24hr": market.get("volume24hr"),
                "bestBid": market.get("bestBid"),
                "bestAsk": market.get("bestAsk"),
                "lastTradePrice": market.get("lastTradePrice"),
            }
        )
    return rows


selected_events = events[: min(len(events), SNAPSHOT_CONFIG["market_sample_size"])]
market_rows = []
for event in selected_events:
    market_rows.extend(extract_market_insight(event))

print(len(market_rows))
pprint(market_rows[:3])

790
[{'active': True,
  'bestAsk': 0.001,
  'bestBid': None,
  'clobTokenIds': ['93592949212798121127213117304912625505836768562433217537850469496310204567695',
                   '3074539347152748632858978545166555332546941892131779352477699494423276162345'],
  'closed': True,
  'conditionId': '0x19ee98e348c0ccb341d1b9566fa14521566e9b2ea7aed34dc407a0ec56be36a2',
  'description': 'This market will resolve to "Yes" if MicroStrategy sells any '
                 'of its Bitcoin by December 31, 2025, 11:59 PM ET. Otherwise, '
                 'this market will resolve to "No".\n'
                 '\n'
                 'The primary resolution source for this market will be '
                 'information from MSTR and on-chain data, however a consensus '
                 'of credible reporting will also be used.',
  'event_id': '16167',
  'event_slug': 'microstrategy-sell-any-bitcoin-in-2025',
  'lastTradePrice': 1,
  'liquidity': None,
  'market_id': '516926',
  'resolutionSource': '',
  '

In [10]:
# 3) Catalogo de tags para segmentar la muestra
tags_response = await polymarket.get_tags(
    query_params={
        "limit": 50,
        "offset": 0,
    }
)

tags_payload = tags_response.json()
tags = normalize_collection(tags_payload, keys=("tags", "results", "data"))

print(type(tags), len(tags))
pprint(tags[:5] if tags else tags)

<class 'list'> 50
[{'createdAt': '2025-02-18T16:58:25.464578Z',
  'id': '101867',
  'label': 'product marekt fit',
  'requiresTranslation': False,
  'slug': 'product-marekt-fit',
  'updatedAt': '2026-04-17T17:23:11.67487Z'},
 {'createdAt': '2024-02-28T22:13:09.166Z',
  'id': '1512',
  'label': 'caitlin clark',
  'publishedAt': '2024-02-28 22:13:06.246+00',
  'requiresTranslation': False,
  'slug': 'caitlin-clark',
  'updatedAt': '2026-04-17T17:23:11.676843Z'},
 {'createdAt': '2024-09-20T00:05:07.009054Z',
  'id': '100601',
  'label': 'virgins',
  'requiresTranslation': False,
  'slug': 'virgins',
  'updatedAt': '2026-04-17T17:23:11.678298Z'},
 {'createdAt': '2024-10-22T03:44:31.174556Z',
  'id': '101025',
  'label': 'Viktoria Plzen',
  'requiresTranslation': False,
  'slug': 'viktoria-plzen',
  'updatedAt': '2026-04-17T17:23:11.679752Z'},
 {'createdAt': '2024-12-10T22:54:59.949671Z',
  'id': '101457',
  'label': 'Jerry After Dark',
  'requiresTranslation': False,
  'slug': 'jerry-after

In [11]:
# 4) Precio actual en batch con CLOB

token_ids = []
for row in market_rows:
    token_ids.extend(row.get("clobTokenIds", []))

token_ids = [token_id for token_id in dict.fromkeys(token_ids) if token_id]
price_payload = [{"token_id": token_id, "side": "BUY"} for token_id in token_ids[:20]]

if price_payload:
    prices_response = await polymarket.post_prices(price_payload)
    prices = prices_response.json()
    pprint(prices)
else:
    print("No se encontraron token ids en la muestra")

if len(token_ids) >= 2:
    batch_history_payload = {
        "markets": token_ids[:20],
        "interval": "1h",
        "fidelity": 1,
    }
    try:
        batch_history_response = await polymarket.post_batch_prices_history(batch_history_payload)
        batch_history = batch_history_response.json()
        pprint(batch_history)
    except Exception as error:
        print(f"No se pudo ejecutar batch-prices-history: {error}")

{'110251828161543119357013227499774714771527179764174739487025581227481937033858': {'BUY': '0.326'},
 '111128191581505463501777127559667396812474366956707382672202929745167742497287': {'BUY': '0.72'},
 '16201530957950630406397949502319734794139620443510795733205872225099141120819': {'BUY': '0.009'},
 '22488814055465212843401871684243737932656687199587127613893702721105723939237': {'BUY': '0.989'},
 '25714007960293389110960044475283546872601238755063051359394740854408462452120': {'BUY': '0.033'},
 '28557614648090529004584076028720900603196666949274543515794672175624115225556': {'BUY': '0.66'},
 '3192689304828767159232889612891719105504357313659012189260030438494464480574': {'BUY': '0.962'},
 '34626184950254225208692030156208941308358060420950772251072421141618169142241': {'BUY': '0.29'},
 '65176388692130651396848427090788038285140833850265294793449655516920659740141': {'BUY': '0.67'},
 '99807503632459517030616292055983105381849115736225256331133222076990620978808': {'BUY': '0.26'}}
{'hi

In [12]:
# 5) Book, historico, trades, holders y activity para un mercado concreto
sample_row = market_rows[0] if market_rows else {}
sample_condition_id = sample_row.get("conditionId")
sample_token = sample_row.get("clobTokenIds", [None])[0] if sample_row.get("clobTokenIds") else None

if sample_token:
    book_response = await polymarket.get_book({"token_id": sample_token})
    book = book_response.json()
    pprint(book)

    history_response = await polymarket.get_prices_history_by_market(
        {
            "market": sample_token,
            "interval": "1h",
            "fidelity": 1,
        }
    )
    history = history_response.json()
    pprint(history)
else:
    print("No se encontro token de muestra para book y prices-history")

sample_wallet = None

if sample_condition_id:
    trades_response = await polymarket.get_trades(
        {
            "market": sample_condition_id,
            "limit": SNAPSHOT_CONFIG["trade_limit"],
            "offset": 0,
        }
    )
    holders_response = await polymarket.get_holders(
        {
            "market": sample_condition_id,
            "limit": SNAPSHOT_CONFIG["holders_limit"],
        }
    )
    trades = trades_response.json()
    holders = holders_response.json()

    pprint(trades[:3] if isinstance(trades, list) else trades)
    pprint(holders[:3] if isinstance(holders, list) else holders)

    if isinstance(trades, list) and trades:
        sample_wallet = trades[0].get("proxyWallet") or trades[0].get("user")
    if not sample_wallet and isinstance(holders, list) and holders:
        sample_wallet = holders[0].get("proxyWallet")

    if sample_wallet:
        activity_response = await polymarket.get_activity(
            {
                "user": sample_wallet,
                "limit": SNAPSHOT_CONFIG["activity_limit"],
                "offset": 0,
                "sortBy": "TIMESTAMP",
                "sortDirection": "DESC",
            }
        )
        activity = activity_response.json()
        pprint(activity[:3] if isinstance(activity, list) else activity)
    else:
        print("No se encontro wallet de ejemplo para activity")
else:
    print("No se encontro conditionId para ejecutar trades/holders/activity")

if OPTIONAL_LOG_ADDRESS:
    try:
        log_params = {"address": OPTIONAL_LOG_ADDRESS}
        if OPTIONAL_LOG_TOPIC:
            log_params["topic0"] = OPTIONAL_LOG_TOPIC
        logs_response = await polygon.get_event_logs_by_address_or_topic(log_params)
        logs = logs_response.json()
        pprint(logs[:3] if isinstance(logs, list) else logs)
    except Exception as error:
        print(f"No se pudo consultar la capa on-chain: {error}")
else:
    print("POLYGON_LOG_ADDRESS no configurado; se omite la capa on-chain")

{'error': 'No orderbook exists for the requested token id'}
{'history': []}
[{'asset': '3074539347152748632858978545166555332546941892131779352477699494423276162345',
  'bio': '',
  'conditionId': '0x19ee98e348c0ccb341d1b9566fa14521566e9b2ea7aed34dc407a0ec56be36a2',
  'eventSlug': 'microstrategy-sell-any-bitcoin-in-2025',
  'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/will-microstrategy-purchase-bitcoin-july-1-7-mzoE5TYk_cCI.jpg',
  'name': 'gem813',
  'outcome': 'No',
  'outcomeIndex': 1,
  'price': 0.999,
  'profileImage': '',
  'profileImageOptimized': '',
  'proxyWallet': '0x42dd06d74eab9c33fa023319e05167f63dbc128a',
  'pseudonym': 'Usable-Preparation',
  'side': 'SELL',
  'size': 1840,
  'slug': 'microstrategy-sell-any-bitcoin-in-2025',
  'timestamp': 1767586869,
  'title': 'MicroStrategy sells any Bitcoin in 2025?',
  'transactionHash': '0xf6c7716b30a529570d5f78556e765b3b89bead3bdd7213a51e3dec1ab574157d'},
 {'asset': '935929492127981211272131173049126255058367685

## Criterios de seleccion para el MVP

De este notebook deberiamos salir con una lista corta de campos y endpoints que realmente aportan senal:

- Gamma: `events`, `markets`, `tags`.
- CLOB: `prices`, `book`, `prices-history`, `batch-prices-history`.
- Data API: `trades`, `holders`, `activity`.
- Polygon: `getLogs` opcional para evidenciar actividad on-chain cuando haya direccion o topic configurados.

Si un campo no cambia ninguna decision de UI o analisis, queda fuera.

Prximo paso recomendado: ejecutar este notebook sobre una muestra de 20 mercados activos y comparar que senales aparecen consistentemente en mercados liquidos vs. mercados poco liquidos.